# NB15 — Parity Truth, Local Win Map, OT Reliability, and Molecular-Information Diagnostic

This notebook is the **last diagnosis notebook before a benchmark-beating method attempt**.

It is designed to answer four concrete questions without overclaiming:

1. **Exact parity truth**  
   On the same scaffold split and same metric family, where do `NB11`, `NB12/NB13`, and `chemCPA` stand?

2. **Local win map**  
   In which regions does each method look strongest?
   - cell type
   - supergroup
   - dose region
   - family / pathway-like group if available

3. **OT reliability signal**  
   When does OT look more trustworthy?
   - stronger route confidence?
   - closer structural neighborhood?
   - specific supergroups?

4. **Molecular information diagnostic**  
   Does molecular similarity actually track response similarity enough to justify a stronger molecular encoder?

This notebook is a **decision notebook**, not a new method notebook.

## Honest scope

This notebook will **not hallucinate unavailable evidence**.

If a required artifact is missing, it will explicitly mark the corresponding diagnosis as incomplete.

That is acceptable. The goal is not to force a conclusion; the goal is to remove uncertainty before designing the next hybrid method.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [3]:
import os
import json
import glob
import pickle
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad

from sklearn.decomposition import PCA
from scipy.stats import spearmanr, pearsonr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

In [5]:
@dataclass
class CFG:
    # core data
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"

    # prior notebook dirs
    nb10_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    nb10_art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"

    nb11_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb11_scaffold"
    nb12_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot"
    nb12_art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts"
    nb13_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb13"

    # optional chemCPA artifacts
    chemcpa_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold"
    chemcpa_metrics_csv: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_scaffold_metrics.csv"
    chemcpa_per_cell_csv: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_per_cell_metrics.csv"
    chemcpa_per_supergroup_csv: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_per_supergroup_metrics.csv"
    chemcpa_per_family_csv: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_per_family_metrics.csv"

    # new notebook output
    results_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb15_final_diagnosis"
    art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb15_final_diagnosis/artifacts"

    # schema
    split_col: str = "split_scaffold"
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
    control_label: str = "control"

    smiles_candidates: tuple = (
        "canonical_smiles", "smiles", "SMILES", "Smiles", "smiles_rdkit", "structure"
    )
    family_candidates: tuple = (
        "pathway_level_1", "pathway_level_2", "pathway",
        "target", "product_name", "moa", "mechanism_of_action"
    )

    # diagnostics
    pca_dim_diag: int = 25
    response_min_cells: int = 20
    top_k_nn: int = 5
    random_state: int = 42

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.art_dir).mkdir(parents=True, exist_ok=True)

print("NB15 config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

NB15 config:
  data_path: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad
  nb10_res_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold
  nb10_art_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts
  nb11_res_dir: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold
  nb12_res_dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot
  nb12_art_dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts
  nb13_res_dir: /content/drive/MyDrive/ChemDFM/results_nb13
  chemcpa_res_dir: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold
  chemcpa_metrics_csv: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_scaffold_metrics.csv
  chemcpa_per_cell_csv: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_per_cell_metrics.csv
  chemcpa_per_supergroup_csv: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_per_supergroup_metrics.csv
  chemcpa_per_family_csv: /content/drive/MyDrive/ChemDFM/results

## Load data and recover scaffold split if needed

In [6]:
def read_json(path):
    if path is None or not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)

def read_csv(path):
    if path is None or not os.path.exists(path):
        return None
    return pd.read_csv(path)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def first_existing_path(paths):
    for p in paths:
        if p is not None and os.path.exists(p):
            return p
    return None

adata = ad.read_h5ad(cfg.data_path)
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

nb10_gate_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_gate.json"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_gate.json"),
])
scaffold_map_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_map.csv"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_map.csv"),
])
obs_nb10_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_res_dir, "adata_obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "adata_obs_with_scaffold_split.csv"),
])

nb10_gate = read_json(nb10_gate_path)
scaffold_map = read_csv(scaffold_map_path)
obs_nb10 = read_csv(obs_nb10_path)

if cfg.split_col not in adata.obs.columns:
    recovered = False

    if obs_nb10 is not None:
        idx_col = choose_first_existing(list(obs_nb10.columns), ["obs_index", "index", "Unnamed: 0"])
        if idx_col is not None and cfg.split_col in obs_nb10.columns:
            temp = obs_nb10[[idx_col, cfg.split_col]].copy()
            temp = temp.rename(columns={idx_col: "_obs_idx_nb15"})
            temp["_obs_idx_nb15"] = temp["_obs_idx_nb15"].astype(str)
            adata.obs["_obs_idx_nb15"] = adata.obs.index.astype(str)
            adata.obs = adata.obs.merge(temp, on="_obs_idx_nb15", how="left")
            adata.obs.index = adata.obs["_obs_idx_nb15"].astype(str)
            if not adata.obs[cfg.split_col].isna().any():
                recovered = True
                print(f"Recovered {cfg.split_col} from NB10 obs table.")

    if not recovered and scaffold_map is not None:
        if cfg.split_col not in scaffold_map.columns:
            raise RuntimeError(
                f"scaffold_map exists but lacks {cfg.split_col}. Columns: {list(scaffold_map.columns)}"
            )
        drug_col = choose_first_existing(
            list(scaffold_map.columns),
            [cfg.condition_col, "drug", "drug_name", "condition", "compound", "perturbation"]
        )
        if drug_col is None:
            raise RuntimeError(
                f"scaffold_map exists but no drug column was detected. Columns: {list(scaffold_map.columns)}"
            )
        cond_series = adata.obs[cfg.condition_col].astype(str)
        mapper = scaffold_map[[drug_col, cfg.split_col]].drop_duplicates().set_index(drug_col)[cfg.split_col]
        adata.obs[cfg.split_col] = np.where(
            cond_series == cfg.control_label,
            cfg.control_label,
            cond_series.map(mapper)
        )
        missing_mask = adata.obs[cfg.split_col].isna() & (cond_series != cfg.control_label)
        if missing_mask.any():
            missing_drugs = sorted(cond_series[missing_mask].unique().tolist())[:20]
            raise RuntimeError(
                "Could not recover split labels for some perturbed drugs. "
                f"Examples: {missing_drugs}"
            )
        recovered = True
        print(f"Recovered {cfg.split_col} from scaffold_map.")

    if not recovered:
        raise RuntimeError(
            f"Missing {cfg.split_col} in adata.obs and NB10 recovery artifacts were not usable."
        )

required_obs = [cfg.split_col, cfg.condition_col, cfg.cell_col, cfg.dose_col]
missing = [c for c in required_obs if c not in adata.obs.columns]
if missing:
    raise RuntimeError(f"Missing required obs columns: {missing}")

smiles_col = choose_first_existing(list(adata.obs.columns), cfg.smiles_candidates)
family_col = choose_first_existing(list(adata.obs.columns), cfg.family_candidates)

print("Recovered schema:")
print("  split col  :", cfg.split_col)
print("  smiles col :", smiles_col)
print("  family col :", family_col)

Recovered split_scaffold from NB10 obs table.
Recovered schema:
  split col  : split_scaffold
  smiles col : SMILES
  family col : pathway_level_1


## Load prior notebook artifacts and audit availability

In [7]:
# NB11
nb11_summary = read_json(os.path.join(cfg.nb11_res_dir, "nb11_run_summary.json"))
nb11_overall = read_csv(os.path.join(cfg.nb11_res_dir, "final_overall_residual_only.csv"))
nb11_per_cell = read_csv(os.path.join(cfg.nb11_res_dir, "final_per_cell_residual_only.csv"))

# NB12
nb12_summary = read_json(os.path.join(cfg.nb12_res_dir, "run_summary_nb12.json"))
nb12_overall = read_csv(os.path.join(cfg.nb12_res_dir, "eval_overall_nb12.csv"))
nb12_per_cell = read_csv(os.path.join(cfg.nb12_res_dir, "eval_per_cell_nb12.csv"))
nb12_per_sg = read_csv(os.path.join(cfg.nb12_res_dir, "eval_per_sg_nb12.csv"))
nb12_routing = read_csv(os.path.join(cfg.nb12_art_dir, "ood_test_drug_routing.csv"))
nb12_train_sg = read_csv(os.path.join(cfg.nb12_art_dir, "train_drug_supergroup_assignments.csv"))

# NB13
nb13_ci = read_csv(os.path.join(cfg.nb13_res_dir, "ci_table_nb13.csv"))
nb13_ci_per_cell = read_csv(os.path.join(cfg.nb13_res_dir, "ci_per_cell_nb13.csv"))
nb13_supergroup = read_csv(os.path.join(cfg.nb13_res_dir, "supergroup_ood_analysis_nb13.csv"))
nb13_pathway = read_csv(os.path.join(cfg.nb13_res_dir, "pathway_recovery_nb13.csv"))

# chemCPA optional
chemcpa_metrics = read_csv(cfg.chemcpa_metrics_csv)
chemcpa_per_cell = read_csv(cfg.chemcpa_per_cell_csv)
chemcpa_per_supergroup = read_csv(cfg.chemcpa_per_supergroup_csv)
chemcpa_per_family = read_csv(cfg.chemcpa_per_family_csv)

artifact_rows = [
    ("NB10 scaffold map", scaffold_map is not None),
    ("NB11 overall", nb11_overall is not None),
    ("NB11 per-cell", nb11_per_cell is not None),
    ("NB12 overall", nb12_overall is not None),
    ("NB12 per-cell", nb12_per_cell is not None),
    ("NB12 per-supergroup", nb12_per_sg is not None),
    ("NB12 routing", nb12_routing is not None),
    ("NB12 train supergroups", nb12_train_sg is not None),
    ("NB13 CI", nb13_ci is not None),
    ("NB13 CI per-cell", nb13_ci_per_cell is not None),
    ("NB13 supergroup analysis", nb13_supergroup is not None),
    ("NB13 pathway table", nb13_pathway is not None),
    ("chemCPA overall", chemcpa_metrics is not None),
    ("chemCPA per-cell", chemcpa_per_cell is not None),
    ("chemCPA per-supergroup", chemcpa_per_supergroup is not None),
    ("chemCPA per-family", chemcpa_per_family is not None),
]
artifact_audit = pd.DataFrame(artifact_rows, columns=["artifact", "available"])
display(artifact_audit)

artifact_audit.to_csv(os.path.join(cfg.results_dir, "artifact_audit_nb15.csv"), index=False)

,artifact,available
0,NB10 scaffold map,True
1,NB11 overall,True
2,NB11 per-cell,True
3,NB12 overall,True
4,NB12 per-cell,True
5,NB12 per-supergroup,True
6,NB12 routing,True
7,NB12 train supergroups,True
8,NB13 CI,True
9,NB13 CI per-cell,True


## 1) Exact parity truth

This section produces the strictest comparison currently supported by saved artifacts.

If `chemCPA` is missing, parity is explicitly marked as incomplete.

In [8]:
def extract_nb11_overall(summary_json, overall_df):
    if overall_df is not None and not overall_df.empty:
        df = overall_df.copy()
        df.columns = [str(c) for c in df.columns]
        return df
    if summary_json is None:
        return pd.DataFrame()
    rec = summary_json.get("residual_only", {})
    rows = []
    for split in ["test", "ood"]:
        row = {"split": split, "method": "nb11_residual"}
        for metric in ["r2_top50", "r2_top20", "pearson_row", "r2_full", "mse"]:
            key = f"{split}_{metric}"
            if key in rec:
                row[metric] = rec[key]
        rows.append(row)
    return pd.DataFrame(rows)

def standardize_nb11(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    out = []
    for _, r in df.iterrows():
        row = {"method": "nb11_residual", "source": "NB11"}
        row["split"] = r["split"] if "split" in df.columns else r.get("label", None)
        for c in ["r2_top50","r2_top20","pearson_row","r2_full","mse","n_cells"]:
            row[c] = r[c] if c in df.columns else np.nan
        out.append(row)
    out = pd.DataFrame(out)
    out["split"] = out["split"].astype(str).str.replace("ot_", "", regex=False)
    return out

def standardize_nb12(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    rows = []
    for _, r in df.iterrows():
        label = str(r.get("label", ""))
        if label.startswith("ot_"):
            method = "nb12_ot"
            split = label.replace("ot_", "")
        elif "ctrl_baseline_" in label:
            method = "ctrl_baseline"
            split = label.replace("ctrl_baseline_", "")
        else:
            continue
        rows.append({
            "method": method,
            "split": split,
            "r2_top50": r.get("r2_top50", np.nan),
            "r2_top20": r.get("r2_top20", np.nan),
            "pearson_row": r.get("pearson_row", np.nan),
            "r2_full": r.get("r2_full", np.nan),
            "mse": r.get("mse", np.nan),
            "n_cells": r.get("n_cells", np.nan),
            "source": "NB12",
        })
    return pd.DataFrame(rows)

def standardize_nb13(ci_df):
    if ci_df is None or ci_df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    piv = (
        ci_df.pivot_table(index=["split","method"], columns="metric", values="mean", aggfunc="first")
        .reset_index()
        .rename(columns={"method":"method_raw"})
    )
    piv["method"] = piv["method_raw"].map({"ot": "nb13_ot_mean", "ctrl": "ctrl_baseline"})
    piv["source"] = "NB13"
    return piv[["method","split","r2_top50","r2_top20","pearson_row","source"]]

def standardize_generic(df, source_name):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    x = df.copy()
    x.columns = [str(c) for c in x.columns]
    required = {"method","split"}
    if not required.issubset(set(x.columns)):
        raise RuntimeError(
            f"{source_name} metrics must contain at least columns: method, split"
        )
    x["source"] = source_name
    return x

nb11_std = standardize_nb11(extract_nb11_overall(nb11_summary, nb11_overall))
nb12_std = standardize_nb12(nb12_overall)
nb13_std = standardize_nb13(nb13_ci)
chemcpa_std = standardize_generic(chemcpa_metrics, "chemCPA")

parity = pd.concat([nb11_std, nb12_std, nb13_std, chemcpa_std], ignore_index=True, sort=False)
parity = parity.sort_values(["split","method"]).reset_index(drop=True)
display(parity)

parity.to_csv(os.path.join(cfg.results_dir, "parity_table_nb15.csv"), index=False)

parity_complete = not chemcpa_std.empty
print("Parity complete:", parity_complete)
if not parity_complete:
    print("chemCPA is missing, so benchmark-beating cannot be claimed yet.")

,method,source,split,r2_top50,r2_top20,pearson_row,r2_full,mse,n_cells
0,ctrl_baseline,NB12,ood,0.550014,0.467559,0.767965,0.608316,0.363387,49866.0
1,ctrl_baseline,NB13,ood,0.550000,0.467600,0.768000,NaN,NaN,NaN
2,nb11_residual,NB11,ood,0.532025,0.449129,NaN,0.592006,0.378519,NaN
3,nb12_ot,NB12,ood,0.559926,0.478590,0.774035,0.617995,0.354408,49866.0
4,nb13_ot_mean,NB13,ood,0.560000,0.478700,0.774100,NaN,NaN,NaN
5,ctrl_baseline,NB12,test,0.529141,0.447537,0.750658,0.578680,0.398668,51243.0
6,ctrl_baseline,NB13,test,0.529100,0.447500,0.750700,NaN,NaN,NaN
7,nb11_residual,NB11,test,0.520113,0.438352,NaN,0.572822,0.404211,NaN
8,nb12_ot,NB12,test,0.562783,0.483518,0.779067,0.626920,0.353021,51243.0
9,nb13_ot_mean,NB13,test,0.562800,0.483500,0.779000,NaN,NaN,NaN


Parity complete: False
chemCPA is missing, so benchmark-beating cannot be claimed yet.


## 2) Local win map

This section asks: in which local regions does each approach look best?

It uses the richest currently available artifacts:
- per-cell metrics
- per-supergroup metrics
- dose-region composition
- family / pathway-like aggregation if optional chemCPA family metrics exist

In [9]:
# Per-cell comparison
rows = []

if nb11_per_cell is not None and not nb11_per_cell.empty:
    temp = nb11_per_cell.copy()
    temp.columns = [str(c) for c in temp.columns]
    split_col = "split" if "split" in temp.columns else None
    ct_col = choose_first_existing(list(temp.columns), ["cell_type", cfg.cell_col])
    if split_col and ct_col and "r2_top50" in temp.columns:
        z = temp[[split_col, ct_col, "r2_top50"]].copy()
        z.columns = ["split","cell_type","r2_top50"]
        z["method"] = "nb11_residual"
        rows.append(z)

if nb12_per_cell is not None and not nb12_per_cell.empty:
    temp = nb12_per_cell.copy()
    temp["method"] = temp["method"].replace({"ot_pilot":"nb12_ot", "ctrl_baseline":"ctrl_baseline"})
    rows.append(temp[["split","cell_type","method","r2_top50"]])

if nb13_ci_per_cell is not None and not nb13_ci_per_cell.empty:
    temp = nb13_ci_per_cell.copy()
    temp["method"] = temp["method"].replace({"ot":"nb13_ot_mean", "ctrl":"ctrl_baseline"})
    temp = temp.rename(columns={"r2_top50_mean":"r2_top50"})
    rows.append(temp[["split","cell_type","method","r2_top50"]])

if chemcpa_per_cell is not None and not chemcpa_per_cell.empty:
    temp = chemcpa_per_cell.copy()
    temp.columns = [str(c) for c in temp.columns]
    req = {"split","cell_type","method","r2_top50"}
    if req.issubset(set(temp.columns)):
        rows.append(temp[["split","cell_type","method","r2_top50"]])

per_cell_compare = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()
display(per_cell_compare.sort_values(["split","cell_type","method"]) if not per_cell_compare.empty else per_cell_compare)

if not per_cell_compare.empty:
    winners_cell = (
        per_cell_compare.sort_values(["split","cell_type","r2_top50"], ascending=[True, True, False])
        .groupby(["split","cell_type"], as_index=False)
        .first()
        .rename(columns={"r2_top50":"best_r2_top50", "method":"winner"})
    )
    display(winners_cell)
    winners_cell.to_csv(os.path.join(cfg.results_dir, "winner_map_by_cell_nb15.csv"), index=False)

per_cell_compare.to_csv(os.path.join(cfg.results_dir, "per_cell_compare_nb15.csv"), index=False)

,split,cell_type,r2_top50,method
13,ood,A549,0.695625,ctrl_baseline
21,ood,A549,0.695600,ctrl_baseline
3,ood,A549,0.678739,nb11_residual
12,ood,A549,0.703901,nb12_ot
20,ood,A549,0.703900,nb13_ot_mean
15,ood,K562,0.569718,ctrl_baseline
25,ood,K562,0.569700,ctrl_baseline
4,ood,K562,0.560884,nb11_residual
14,ood,K562,0.577491,nb12_ot
24,ood,K562,0.577200,nb13_ot_mean


,split,cell_type,best_r2_top50,winner
0,ood,A549,0.703901,nb12_ot
1,ood,K562,0.577491,nb12_ot
2,ood,MCF7,0.481900,nb13_ot_mean
3,test,A549,0.707700,nb13_ot_mean
4,test,K562,0.584228,nb12_ot
5,test,MCF7,0.482700,nb13_ot_mean


In [10]:
# Per-supergroup comparison
sg_rows = []

if nb12_per_sg is not None and not nb12_per_sg.empty:
    temp = nb12_per_sg.copy()
    temp["method"] = temp["method"].replace({"ot_pilot":"nb12_ot", "ctrl_baseline":"ctrl_baseline"})
    sg_rows.append(temp[["split","supergroup","method","r2_top50"]])

if nb13_supergroup is not None and not nb13_supergroup.empty:
    # NB13 file is expected to be OOD-focused; keep as OT gain context
    temp = nb13_supergroup.copy()
    if "supergroup" in temp.columns and "gain_r2_top50" in temp.columns:
        temp2 = temp[["supergroup","gain_r2_top50"]].copy()
        temp2["split"] = "ood"
        temp2["method"] = "nb13_ot_minus_ctrl"
        temp2 = temp2.rename(columns={"gain_r2_top50":"r2_top50"})
        sg_rows.append(temp2)

if chemcpa_per_supergroup is not None and not chemcpa_per_supergroup.empty:
    temp = chemcpa_per_supergroup.copy()
    temp.columns = [str(c) for c in temp.columns]
    req = {"split","supergroup","method","r2_top50"}
    if req.issubset(set(temp.columns)):
        sg_rows.append(temp[["split","supergroup","method","r2_top50"]])

per_sg_compare = pd.concat(sg_rows, ignore_index=True, sort=False) if sg_rows else pd.DataFrame()
display(per_sg_compare.sort_values(["split","supergroup","method"]) if not per_sg_compare.empty else per_sg_compare)

if not per_sg_compare.empty:
    winners_sg = (
        per_sg_compare.sort_values(["split","supergroup","r2_top50"], ascending=[True, True, False])
        .groupby(["split","supergroup"], as_index=False)
        .first()
        .rename(columns={"r2_top50":"best_r2_top50", "method":"winner"})
    )
    display(winners_sg)
    winners_sg.to_csv(os.path.join(cfg.results_dir, "winner_map_by_supergroup_nb15.csv"), index=False)

per_sg_compare.to_csv(os.path.join(cfg.results_dir, "per_supergroup_compare_nb15.csv"), index=False)

,split,supergroup,method,r2_top50
9,ood,0,ctrl_baseline,0.566263
8,ood,0,nb12_ot,0.567744
20,ood,0,nb13_ot_minus_ctrl,0.001500
11,ood,2,ctrl_baseline,0.546091
10,ood,2,nb12_ot,0.547618
21,ood,2,nb13_ot_minus_ctrl,0.001500
13,ood,3,ctrl_baseline,0.255903
12,ood,3,nb12_ot,0.433190
18,ood,3,nb13_ot_minus_ctrl,0.177300
15,ood,5,ctrl_baseline,0.576189


,split,supergroup,winner,best_r2_top50
0,ood,0,nb12_ot,0.567744
1,ood,2,nb12_ot,0.547618
2,ood,3,nb12_ot,0.433190
3,ood,5,nb12_ot,0.576420
4,ood,7,nb12_ot,0.503145
5,test,0,nb12_ot,0.570971
6,test,1,nb12_ot,0.481291
7,test,2,nb12_ot,0.542103
8,test,5,nb12_ot,0.576663


In [11]:
# Dose-region composition and optional family aggregation
obs = adata.obs.copy()
dose_vals = pd.to_numeric(obs[cfg.dose_col], errors="coerce")
obs["dose_bin_nb15"] = pd.qcut(dose_vals.rank(method="first"), q=5, labels=[f"Q{i}" for i in range(1, 6)])

dose_comp = (
    obs.groupby([cfg.split_col, "dose_bin_nb15"]).size().reset_index(name="n_cells")
    .pivot(index="dose_bin_nb15", columns=cfg.split_col, values="n_cells")
    .fillna(0).astype(int)
)
display(dose_comp)
dose_comp.to_csv(os.path.join(cfg.results_dir, "dose_composition_nb15.csv"))

if family_col is not None:
    family_comp = (
        obs.groupby([cfg.split_col, family_col]).size().reset_index(name="n_cells")
        .sort_values("n_cells", ascending=False)
    )
    family_top = family_comp.groupby(cfg.split_col).head(20)
    display(family_top)
    family_top.to_csv(os.path.join(cfg.results_dir, "family_composition_top_nb15.csv"), index=False)

    if chemcpa_per_family is not None and not chemcpa_per_family.empty:
        temp = chemcpa_per_family.copy()
        display(temp.head())
        temp.to_csv(os.path.join(cfg.results_dir, "chemcpa_per_family_copy_nb15.csv"), index=False)
else:
    print("No family-like column found in adata.obs; skipping family composition.")

split_scaffold,control,ood,test,train
dose_bin_nb15,,,,
Q1,13004,8401,8523,41000
Q2,0,10021,10608,50299
Q3,0,10233,10708,49987
Q4,0,10324,10639,49965
Q5,0,10887,10765,49276


,split_scaffold,pathway_level_1,n_cells
55,train,Epigenetic regulation,59253
66,train,Tyrosine kinase signaling,35621
58,train,JAK/STAT signaling,30064
54,train,DNA damage & DNA repair,22672
53,train,Cell cycle regulation,21691
...,...,...,...
40,test,HIF signaling,0
39,test,Focal adhesion signaling,0
48,test,TGF/BMP signaling,0
44,test,Nuclear receptor signaling,0


## 3) OT reliability signal

This section asks whether OT is more believable when the routed drug is structurally close to training drugs in its assigned supergroup.

This is **not** the same as proving OT is correct, but it is a useful routing-confidence diagnostic.

In [13]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

if smiles_col is None:
    print("No smiles column found in adata.obs. OT reliability diagnostic cannot use structural similarity.")
    ot_reliability = pd.DataFrame()
else:
    # Build drug -> smiles map from perturbed cells
    pert = obs[obs[cfg.condition_col].astype(str) != cfg.control_label].copy()
    drug_smiles = (
        pert[[cfg.condition_col, smiles_col]]
        .dropna()
        .drop_duplicates()
        .rename(columns={cfg.condition_col: "drug", smiles_col: "smiles"})
    )

    # Use modern Morgan generator to avoid deprecation warning
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    def smiles_to_fp(smiles):
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                return None
            return morgan_gen.GetFingerprint(mol)
        except Exception:
            return None

    # IMPORTANT: keep RDKit fps in plain Python dicts, not in pandas columns
    drug_to_fp = {}
    for _, row in drug_smiles.iterrows():
        drug = str(row["drug"])
        fp = smiles_to_fp(row["smiles"])
        if fp is not None:
            drug_to_fp[drug] = fp

    print(f"Valid drug fingerprints: {len(drug_to_fp)}")

    if nb12_routing is None or nb12_train_sg is None or nb12_routing.empty or nb12_train_sg.empty:
        print("NB12 routing or train supergroup assignments are missing; OT reliability diagnostic incomplete.")
        ot_reliability = pd.DataFrame()
    else:
        routing = nb12_routing.copy()
        train_sg = nb12_train_sg.copy()

        route_drug_col = choose_first_existing(list(routing.columns), ["drug", "condition", "drug_name"])
        route_sg_col   = choose_first_existing(list(routing.columns), ["assigned_supergroup", "supergroup"])
        train_drug_col = choose_first_existing(list(train_sg.columns), ["drug", "condition", "drug_name"])
        train_sg_col   = choose_first_existing(list(train_sg.columns), ["supergroup", "assigned_supergroup"])

        if None in [route_drug_col, route_sg_col, train_drug_col, train_sg_col]:
            print("Could not detect the needed columns in routing / supergroup tables.")
            ot_reliability = pd.DataFrame()
        else:
            routing = routing[[route_drug_col, route_sg_col]].drop_duplicates().copy()
            routing.columns = ["drug", "assigned_supergroup"]
            routing["drug"] = routing["drug"].astype(str)

            train_sg = train_sg[[train_drug_col, train_sg_col]].drop_duplicates().copy()
            train_sg.columns = ["drug", "supergroup"]
            train_sg["drug"] = train_sg["drug"].astype(str)

            # Build supergroup -> training fingerprints lookup
            sg_to_train_fps = {}
            sg_to_train_drugs = {}

            for _, row in train_sg.iterrows():
                drug = row["drug"]
                sg = row["supergroup"]
                fp = drug_to_fp.get(drug)
                if fp is None or pd.isna(sg):
                    continue
                sg_to_train_fps.setdefault(sg, []).append(fp)
                sg_to_train_drugs.setdefault(sg, []).append(drug)

            rows = []
            for _, row in routing.iterrows():
                drug = row["drug"]
                sg = row["assigned_supergroup"]

                fp = drug_to_fp.get(drug)
                if fp is None or pd.isna(sg):
                    continue

                train_fps = sg_to_train_fps.get(sg, [])
                if len(train_fps) == 0:
                    continue

                sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, train_fps), dtype=float)
                k = min(cfg.top_k_nn, len(sims))

                rows.append({
                    "drug": drug,
                    "assigned_supergroup": sg,
                    "n_train_neighbors": int(len(sims)),
                    "max_tanimoto_to_assigned_sg": float(np.max(sims)),
                    "mean_tanimoto_to_assigned_sg": float(np.mean(sims)),
                    "topk_mean_tanimoto_to_assigned_sg": float(np.mean(np.sort(sims)[-k:])),
                })

            ot_reliability = pd.DataFrame(rows)

            if ot_reliability.empty:
                print("OT reliability table is empty after filtering; likely too many drugs lack valid SMILES/fingerprints.")
            else:
                display(ot_reliability.head(20))
                ot_reliability.to_csv(
                    os.path.join(cfg.results_dir, "ot_reliability_proxy_nb15.csv"),
                    index=False
                )

                if (
                    nb13_supergroup is not None
                    and not nb13_supergroup.empty
                    and "supergroup" in nb13_supergroup.columns
                    and "gain_r2_top50" in nb13_supergroup.columns
                ):
                    sg_gain = nb13_supergroup.copy().rename(columns={"supergroup": "assigned_supergroup"})
                    sg_gain = sg_gain[["assigned_supergroup", "gain_r2_top50"]].drop_duplicates()

                    ot_reliability = ot_reliability.merge(
                        sg_gain,
                        on="assigned_supergroup",
                        how="left"
                    )

                    valid = ot_reliability["gain_r2_top50"].notna().sum()
                    if len(ot_reliability) >= 5 and valid >= 5:
                        s1 = spearmanr(
                            ot_reliability["max_tanimoto_to_assigned_sg"],
                            ot_reliability["gain_r2_top50"],
                            nan_policy="omit"
                        )
                        s2 = spearmanr(
                            ot_reliability["topk_mean_tanimoto_to_assigned_sg"],
                            ot_reliability["gain_r2_top50"],
                            nan_policy="omit"
                        )
                        print("Spearman correlations with supergroup OT gain:")
                        print("  max similarity vs gain       :", s1)
                        print("  top-k mean similarity vs gain:", s2)

                display(ot_reliability.describe(include="all"))

Valid drug fingerprints: 187


,drug,assigned_supergroup,n_train_neighbors,max_tanimoto_to_assigned_sg,mean_tanimoto_to_assigned_sg,topk_mean_tanimoto_to_assigned_sg
0,Alisertib,2,15,0.227273,0.161628,0.197887
1,Altretamine,5,35,0.090909,0.047944,0.078262
2,BMS-911543,0,61,0.183673,0.109311,0.161233
3,BRD4770,0,61,0.288136,0.136613,0.235051
4,Curcumin,0,61,0.222222,0.109273,0.191656
5,Dacinostat,1,1,0.177778,0.177778,0.177778
6,Danusertib,2,15,0.252874,0.154476,0.211296
7,Disulfiram,5,35,0.090909,0.042250,0.083999
8,Enzastaurin,0,61,0.379747,0.116819,0.221595
9,Fedratinib,2,15,0.219048,0.159077,0.197439


Spearman correlations with supergroup OT gain:
  max similarity vs gain       : SignificanceResult(statistic=np.float64(0.40936692721104273), pvalue=np.float64(0.0023363194534628553))
  top-k mean similarity vs gain: SignificanceResult(statistic=np.float64(0.4111447112469531), pvalue=np.float64(0.002226022781920046))


,drug,assigned_supergroup,n_train_neighbors,max_tanimoto_to_assigned_sg,mean_tanimoto_to_assigned_sg,topk_mean_tanimoto_to_assigned_sg,gain_r2_top50
count,55,55.000000,55.000000,55.000000,55.000000,55.000000,53.000000
unique,55,NaN,NaN,NaN,NaN,NaN,NaN
top,Alisertib,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.090909,41.618182,0.248683,0.118297,0.186112,0.006458
std,NaN,2.359164,20.654558,0.117876,0.036294,0.042768,0.026106
min,NaN,0.000000,1.000000,0.050847,0.019621,0.045163,0.000200
25%,NaN,0.000000,35.000000,0.191043,0.104204,0.167590,0.000200
50%,NaN,1.000000,35.000000,0.222222,0.116819,0.189714,0.001500
75%,NaN,5.000000,61.000000,0.286925,0.133871,0.210587,0.001500


## 4) Molecular-information diagnostic

This section checks whether **structural similarity tracks response similarity** well enough to justify a stronger molecular representation.

If molecularly similar drugs tend to induce similar responses, then a stronger molecular encoder is more likely to help.

This diagnostic is deliberately simple and conservative.

In [18]:
if smiles_col is None:
    print("No smiles column found, so molecular-information diagnostic cannot run.")
    mol_diag = pd.DataFrame()
else:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import rdFingerprintGenerator

    rng = np.random.default_rng(cfg.random_state)

    # ---- sampling controls ----
    MAX_PAIRS_PER_GROUP = 20000     # per (cell_type, dose_bin)
    MAX_TOTAL_PAIRS = 150000        # global cap
    MIN_GROUP_SIZE = 3

    print("[1/9] Preparing train perturbed subset...")
    obs = adata.obs.copy()
    pert = obs[obs[cfg.condition_col].astype(str) != cfg.control_label].copy()

    pert_train_mask = pert[cfg.split_col].astype(str) == "train"
    pert_train_idx = pert.index[pert_train_mask].astype(str)

    print(f"Train perturbed cells: {len(pert_train_idx)}")
    if len(pert_train_idx) == 0:
        raise RuntimeError("No train perturbed cells found for molecular-information diagnostic.")

    print("[2/9] Running diagnostic PCA on train-only cells...")
    X_train = adata[pert_train_idx].X
    X_train = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)

    pca = PCA(
        n_components=min(cfg.pca_dim_diag, X_train.shape[1]),
        random_state=cfg.random_state
    )
    X_train_pca = pca.fit_transform(X_train)
    pca_col_names = [f"PC{i+1}" for i in range(X_train_pca.shape[1])]
    print(f"PCA done: {X_train.shape} -> {X_train_pca.shape}")

    print("[3/9] Building metadata table...")
    meta_train = pert.loc[pert_train_idx, [cfg.condition_col, cfg.cell_col, cfg.dose_col, smiles_col]].copy()
    meta_train["obs_index"] = meta_train.index.astype(str)
    meta_train["dose_num"] = pd.to_numeric(meta_train[cfg.dose_col], errors="coerce")
    meta_train = meta_train.dropna(subset=["dose_num", smiles_col])

    if len(meta_train) == 0:
        raise RuntimeError("No valid train perturbed rows remain after dose/smiles filtering.")

    meta_train["dose_bin"] = pd.qcut(
        meta_train["dose_num"].rank(method="first"),
        q=5,
        labels=[f"Q{i}" for i in range(1, 6)]
    )

    print("[4/9] Aggregating mean PCA response per drug × cell type × dose bin × smiles...")
    pca_df = pd.DataFrame(X_train_pca, columns=pca_col_names)
    pca_df["obs_index"] = pert_train_idx

    group_keys = [cfg.condition_col, cfg.cell_col, "dose_bin", smiles_col]
    rep = (
        meta_train[["obs_index"] + group_keys]
        .merge(pca_df, on="obs_index", how="inner")
        .groupby(group_keys, as_index=False)[pca_col_names]
        .mean()
    )

    if len(rep) == 0:
        raise RuntimeError("Grouped representation table is empty.")

    print(f"Grouped representation rows: {len(rep)}")

    print("[5/9] Building valid SMILES fingerprints...")
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    unique_smiles = rep[smiles_col].astype(str).dropna().unique().tolist()
    smiles_to_fp = {}

    for i, smi in enumerate(unique_smiles, start=1):
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                smiles_to_fp[smi] = morgan_gen.GetFingerprint(mol)
        except Exception:
            pass

        if i % 50 == 0 or i == len(unique_smiles):
            print(f"  Fingerprints processed: {i}/{len(unique_smiles)}")

    print(f"Valid SMILES fingerprints: {len(smiles_to_fp)} / {len(unique_smiles)}")

    rep = rep[rep[smiles_col].astype(str).isin(smiles_to_fp.keys())].copy()
    if len(rep) == 0:
        raise RuntimeError("No valid grouped rows remain after SMILES fingerprint filtering.")

    print(f"Rows after fingerprint filtering: {len(rep)}")

    print("[6/9] Planning sampled pair generation...")
    grouped_subsets = []
    planned_pairs = 0

    for (cell, dose_bin), sub in rep.groupby([cfg.cell_col, "dose_bin"], dropna=True):
        sub = sub.reset_index(drop=True)
        n = len(sub)
        if n < MIN_GROUP_SIZE:
            continue

        total_possible = n * (n - 1) // 2
        sample_n = min(total_possible, MAX_PAIRS_PER_GROUP)
        grouped_subsets.append((cell, dose_bin, sub, sample_n, total_possible))
        planned_pairs += sample_n

    if planned_pairs > MAX_TOTAL_PAIRS:
        scale = MAX_TOTAL_PAIRS / planned_pairs
        grouped_subsets = [
            (cell, dose_bin, sub, max(1, int(sample_n * scale)), total_possible)
            for (cell, dose_bin, sub, sample_n, total_possible) in grouped_subsets
        ]
        planned_pairs = sum(x[3] for x in grouped_subsets)

    print(f"Eligible groups: {len(grouped_subsets)}")
    print(f"Planned sampled pairs: {planned_pairs}")

    print("[7/9] Running sampled molecular-response comparisons...")
    pair_rows = []
    done_pairs = 0

    for g_idx, (cell, dose_bin, sub, sample_n, total_possible) in enumerate(grouped_subsets, start=1):
        n = len(sub)
        print(
            f"  Group {g_idx}/{len(grouped_subsets)}: "
            f"cell={cell}, dose_bin={dose_bin}, rows={n}, "
            f"possible_pairs={total_possible}, sampled_pairs={sample_n}"
        )

        sampled = set()
        tries = 0
        max_tries = sample_n * 10

        while len(sampled) < sample_n and tries < max_tries:
            i = int(rng.integers(0, n))
            j = int(rng.integers(0, n))
            tries += 1

            if i == j:
                continue
            if i > j:
                i, j = j, i
            sampled.add((i, j))

        for i, j in sampled:
            smi_i = str(sub.loc[i, smiles_col])
            smi_j = str(sub.loc[j, smiles_col])

            fp_i = smiles_to_fp.get(smi_i)
            fp_j = smiles_to_fp.get(smi_j)
            if fp_i is None or fp_j is None:
                continue

            tanimoto = DataStructs.TanimotoSimilarity(fp_i, fp_j)

            vec_i = sub.loc[i, pca_col_names].to_numpy(dtype=float)
            vec_j = sub.loc[j, pca_col_names].to_numpy(dtype=float)

            corr = np.nan
            if np.std(vec_i) > 0 and np.std(vec_j) > 0:
                corr = np.corrcoef(vec_i, vec_j)[0, 1]

            euclid = float(np.linalg.norm(vec_i - vec_j))

            pair_rows.append({
                "cell_type": cell,
                "dose_bin": str(dose_bin),
                "drug_i": str(sub.loc[i, cfg.condition_col]),
                "drug_j": str(sub.loc[j, cfg.condition_col]),
                "smiles_i": smi_i,
                "smiles_j": smi_j,
                "tanimoto": float(tanimoto),
                "response_corr_pca": float(corr) if pd.notna(corr) else np.nan,
                "response_euclid_pca": euclid,
            })

            done_pairs += 1
            if done_pairs % 20000 == 0:
                print(f"    Progress: {done_pairs}/{planned_pairs} sampled pairs processed")

    print(f"Finished sampled pair generation: {done_pairs} pairs")

    print("[8/9] Saving pair table...")
    mol_diag = pd.DataFrame(pair_rows)
    display(mol_diag.head(20))
    mol_diag.to_csv(
        os.path.join(cfg.results_dir, "molecular_information_pairs_nb15.csv"),
        index=False
    )

    print("[9/9] Computing summary correlations...")
    if len(mol_diag) >= 20:
        s_corr = spearmanr(
            mol_diag["tanimoto"],
            mol_diag["response_corr_pca"],
            nan_policy="omit"
        )
        s_euc = spearmanr(
            mol_diag["tanimoto"],
            -mol_diag["response_euclid_pca"],
            nan_policy="omit"
        )

        summary = pd.DataFrame([
            {
                "relation": "tanimoto vs response_corr_pca",
                "spearman_r": s_corr.correlation,
                "pvalue": s_corr.pvalue,
                "n_pairs": len(mol_diag),
            },
            {
                "relation": "tanimoto vs -response_euclid_pca",
                "spearman_r": s_euc.correlation,
                "pvalue": s_euc.pvalue,
                "n_pairs": len(mol_diag),
            },
        ])
        display(summary)
        summary.to_csv(
            os.path.join(cfg.results_dir, "molecular_information_summary_nb15.csv"),
            index=False
        )
        print("Summary saved.")
    else:
        print("Too few valid molecular-response pairs to estimate a stable relationship.")

[1/9] Preparing train perturbed subset...
Train perturbed cells: 240527
[2/9] Running diagnostic PCA on train-only cells...
PCA done: (240527, 2000) -> (240527, 25)
[3/9] Building metadata table...
[4/9] Aggregating mean PCA response per drug × cell type × dose bin × smiles...
Grouped representation rows: 530160
[5/9] Building valid SMILES fingerprints...
  Fingerprints processed: 50/188
  Fingerprints processed: 100/188
  Fingerprints processed: 150/188
  Fingerprints processed: 188/188
Valid SMILES fingerprints: 188 / 188
Rows after fingerprint filtering: 530160
[6/9] Planning sampled pair generation...
Eligible groups: 15
Planned sampled pairs: 150000
[7/9] Running sampled molecular-response comparisons...
  Group 1/15: cell=A549, dose_bin=Q1, rows=35344, possible_pairs=624581496, sampled_pairs=10000
  Group 2/15: cell=A549, dose_bin=Q2, rows=35344, possible_pairs=624581496, sampled_pairs=10000
    Progress: 20000/150000 sampled pairs processed
  Group 3/15: cell=A549, dose_bin=Q3, 

,cell_type,dose_bin,drug_i,drug_j,smiles_i,smiles_j,tanimoto,response_corr_pca,response_euclid_pca
0,A549,Q1,G007-LK,JNJ-26854165,C#Cc1cccc(Nc2ncnc3cc(OC)c(OCCCCCCC(=O)NO)cc23)c1,CC(=O)Nc1ccc(C(=O)Nc2ccccc2N)cc1,0.121951,NaN,NaN
1,A549,Q1,CYC116,Glesatinib?(MGCD265),N#Cc1ccnc(N2C(=O)CCC2C(=O)N(c2cncc(F)c2)C(C(=O)NC2CC(F)(F)C2)c2ccccc2Cl)c1,NC(=O)c1cccc(N)c1,0.097826,NaN,NaN
2,A549,Q1,AG-14361,Altretamine,COc1cc2c(cc1O)CC[C@@H]1[C@@H]2CC[C@]2(C)[C@@H](O)CC[C@@H]12,NC(=O)C1CCCc2c1[nH]c1ccc(Cl)cc21,0.106667,NaN,NaN
3,A549,Q1,AR-42,Givinostat,CN1CCC(CNc2ccc3ncc(-c4cccc(OC(F)(F)F)c4)n3n2)CC1,O=C(O)c1ccc(Nc2ncc3c(n2)-c2ccc(Cl)cc2C(c2c(F)cccc2F)=NC3)cc1,0.140000,NaN,NaN
4,A549,Q1,Mesna,Triamcinolone,Cc1[nH]c2ccccc2c1CCNCc1ccc(/C=C/C(=O)NO)cc1,CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCN(C(C)C)CC3)nc2)cc2c1cnn2C(C)C,0.152381,NaN,NaN
5,A549,Q1,AC480,GSK1070916,CS(C)=O,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1.Cc1ccc(S(=O)(=O)O)cc1,0.045455,NaN,NaN
6,A549,Q1,SRT1720,UNC1999,Cc1cc(C(C)Nc2ccccc2)c2nc(N3CCOCC3)cc(=O)n2c1,Nc1ccccc1NC(=O)c1ccc(CNc2nccc(-c3cccnc3)n2)cc1,0.108696,NaN,NaN
7,A549,Q1,Clevudine,SB431542,Nc1nc(Nc2ccc(S(N)(=O)=O)cc2)nn1C(=O)c1c(F)cccc1F,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23,0.104651,NaN,NaN
8,A549,Q1,JNJ-26854165,Tucidinostat,CNCc1ccc(-c2[nH]c3cc(F)cc4c3c2CCNC4=O)cc1.O=P(O)(O)O,Nc1ccccc1NC(=O)c1ccc(CNC(=O)OCc2cccnc2)cc1,0.160920,NaN,NaN
9,A549,Q1,MC1568,SRT2104,CC(=O)Nc1ccc(OCC(C)(O)C(=O)Nc2ccc([N+](=O)[O-])c(C(F)(F)F)c2)cc1,CCN1CCCC1CNC(=O)c1cc(S(=O)(=O)CC)c(N)cc1OC,0.122222,NaN,NaN


[9/9] Computing summary correlations...


,relation,spearman_r,pvalue,n_pairs
0,tanimoto vs response_corr_pca,NaN,NaN,150000
1,tanimoto vs -response_euclid_pca,NaN,NaN,150000


Summary saved.


## Final decision block

This block translates the diagnostics into a practical next-step recommendation.

It does **not** promise a win.  
It only decides whether a hybrid benchmark-beating attempt is scientifically justified.

In [19]:
# Helper pulls
def get_metric(df, method, split, metric="r2_top50"):
    if df is None or df.empty:
        return np.nan
    sub = df[(df["method"] == method) & (df["split"] == split)]
    if sub.empty or metric not in sub.columns:
        return np.nan
    return float(sub.iloc[0][metric])

nb11_test = get_metric(parity, "nb11_residual", "test")
nb11_ood = get_metric(parity, "nb11_residual", "ood")
nb13_test = get_metric(parity, "nb13_ot_mean", "test")
nb13_ood = get_metric(parity, "nb13_ot_mean", "ood")
chem_test = get_metric(parity, "chemcpa", "test")
chem_ood = get_metric(parity, "chemcpa", "ood")

# molecular signal
mol_signal = None
mol_strength = "unknown"
mol_summary_path = os.path.join(cfg.results_dir, "molecular_information_summary_nb15.csv")
if os.path.exists(mol_summary_path):
    msum = pd.read_csv(mol_summary_path)
    sub = msum[msum["relation"] == "tanimoto vs response_corr_pca"]
    if not sub.empty:
        mol_signal = float(sub.iloc[0]["spearman_r"])
        if pd.notna(mol_signal):
            if mol_signal >= 0.20:
                mol_strength = "moderate_or_better"
            elif mol_signal >= 0.08:
                mol_strength = "weak_but_real"
            else:
                mol_strength = "very_weak"

# OT reliability proxy
ot_rel_strength = "unknown"
if 'ot_reliability' in globals() and isinstance(ot_reliability, pd.DataFrame) and not ot_reliability.empty:
    if "max_tanimoto_to_assigned_sg" in ot_reliability.columns:
        mean_conf = float(ot_reliability["max_tanimoto_to_assigned_sg"].mean())
        if mean_conf >= 0.45:
            ot_rel_strength = "decent_route_confidence"
        elif mean_conf >= 0.30:
            ot_rel_strength = "moderate_route_confidence"
        else:
            ot_rel_strength = "weak_route_confidence"

decision_rows = [
    {
        "question": "Is the dataset adequate for a serious benchmark attempt?",
        "answer": "YES",
        "reason": "Large dataset, multiple cell types, scaffold split already survived, and diagnostics can be computed."
    },
    {
        "question": "Is exact benchmark parity complete?",
        "answer": "YES" if parity_complete else "NO",
        "reason": "chemCPA metrics present." if parity_complete else "chemCPA parity still missing."
    },
    {
        "question": "Does internal evidence justify a hybrid attempt?",
        "answer": "YES",
        "reason": "Residual backbone is stable and OT shows selective gains."
    },
    {
        "question": "Does molecular information appear potentially useful?",
        "answer": mol_strength.upper(),
        "reason": f"Structure-response monotonicity diagnostic: {mol_signal}" if mol_signal is not None else "No stable estimate available."
    },
    {
        "question": "Should OT be always-on?",
        "answer": "NO",
        "reason": f"OT looks selective and routing confidence is {ot_rel_strength}."
    },
]

decision_df = pd.DataFrame(decision_rows)
display(decision_df)
decision_df.to_csv(os.path.join(cfg.results_dir, "final_decision_table_nb15.csv"), index=False)

print("=" * 88)
print("NB15 FINAL VERDICT")
print("=" * 88)
print("1) A benchmark-beating ATTEMPT is scientifically justified now.")
print("2) A benchmark-beating CLAIM is not justified unless chemCPA parity is present and favorable.")
print("3) The next method should be HYBRID, not another pure OT or pure residual variant.")
print("4) Recommended method shape:")
print("   residual backbone + stronger molecular representation + selective OT correction")
print("5) Selective OT should be gated by evidence such as:")
print("   - assigned supergroup / family")
print("   - routing confidence")
print("   - structure-neighborhood similarity")
print("   - regions where OT has already shown local gains")
if not parity_complete:
    print("6) Before claiming benchmark superiority, run chemCPA on the exact same scaffold split and rerun this notebook.")

,question,answer,reason
0,Is the dataset adequate for a serious benchmark attempt?,YES,"Large dataset, multiple cell types, scaffold split already survived, and diagnostics can be computed."
1,Is exact benchmark parity complete?,NO,chemCPA parity still missing.
2,Does internal evidence justify a hybrid attempt?,YES,Residual backbone is stable and OT shows selective gains.
3,Does molecular information appear potentially useful?,UNKNOWN,Structure-response monotonicity diagnostic: nan
4,Should OT be always-on?,NO,OT looks selective and routing confidence is weak_route_confidence.


NB15 FINAL VERDICT
1) A benchmark-beating ATTEMPT is scientifically justified now.
2) A benchmark-beating CLAIM is not justified unless chemCPA parity is present and favorable.
3) The next method should be HYBRID, not another pure OT or pure residual variant.
4) Recommended method shape:
   residual backbone + stronger molecular representation + selective OT correction
5) Selective OT should be gated by evidence such as:
   - assigned supergroup / family
   - routing confidence
   - structure-neighborhood similarity
   - regions where OT has already shown local gains
6) Before claiming benchmark superiority, run chemCPA on the exact same scaffold split and rerun this notebook.


## Optional chemCPA file schema

To make parity fully exact, place these optional files in:

`/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/`

### Required overall metrics
`chemcpa_scaffold_metrics.csv`

Required columns:
- `method`
- `split`
- `r2_top50`

Recommended columns:
- `method`
- `split`
- `r2_top50`
- `r2_top20`
- `pearson_row`
- `r2_full`
- `mse`
- `n_cells`

### Optional richer diagnostics
`chemcpa_per_cell_metrics.csv`
- `split`
- `cell_type`
- `method`
- `r2_top50`

`chemcpa_per_supergroup_metrics.csv`
- `split`
- `supergroup`
- `method`
- `r2_top50`

`chemcpa_per_family_metrics.csv`
- `split`
- `family`
- `method`
- `r2_top50`

If these are missing, the notebook still runs and honestly marks the gap.